In [1]:
# ! brew install git-lfs
# ! git-lfs clone https://github.com/amazon-science/esci-data.git
# ! brew install uv
# ! uv venv --python 3.12
# ! uv pip install pandas requests pyarrow

In [2]:
import requests
import json
import pandas as pd

In [3]:
# OpenSearchの接続情報
BASE_URL = "http://localhost:9200"
INDEX_NAME = "amazon-jp"
HEADERS = {"Content-Type": "application/json"}

## Amazonデータセットの閲覧

In [4]:
df_products = pd.read_parquet(
    "esci-data/shopping_queries_dataset/shopping_queries_dataset_products.parquet",
    filters=[("product_locale", "=", "jp")],
)
df_products.sample(n=5, random_state=0)

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
178457,B07Z8LLPJ7,TAFLY シール 恐竜きょうりゅうシール ごほうびシール 生き物シール学校幼稚園保育園習シ...,多用途：<br> お手紙やメッセージカード、写真も華やかにデコレーションできるシールです。<...,ノートの整理、お手紙、メッセージカード作り、スクラップブッキング、プレゼントなどに…。\n一...,TAFLY,NaN,jp
221268,4086179083,ジョジョの奇妙な冒険 1~7巻(第1・2部)セット (集英社文庫(コミック版)),NaN,NaN,集英社,NaN,jp
116995,B01CPBVGLS,【輸入王オリジナル】時計パーツ ブライトリングと互換性あり Dバックル用 ベルト 型押しクロ...,高級時計の維持費の高さに悩んでいませんか？消耗品のパーツやメンテナンスで所有しているだけで維...,素材：カーフ※Dバックルは付属しません。 純正のDバックルにもお使いいただけます。\n時計側...,輸入王オリジナル,NaN,jp
289872,B000GPIDLC,ザ･ヴェルヴェット･ロープ･ツアー･ライブ [DVD],NaN,NaN,コロムビアミュージックエンタテインメント,NaN,jp
335136,B08R9YS3LT,NHK いないいないばあっ! ~ワンワン☆ダンス~〔CD〕,NaN,NaN,コロムビアミュージックエンタテインメント,NaN,jp


## OpenSearchへのデータ登録

In [7]:
mapping = {
    "mappings": {
        "properties": {
            "product_id": {"type": "keyword"},
            "product_title": {"type": "text"},
            "product_bullet_point": {"type": "text"},
            "product_description": {"type": "text"},
            "product_brand": {"type": "text"},
            "product_color": {"type": "text"},
            "product_locale": {"type": "keyword"},
        }
    }
}
# requests.delete(f"{BASE_URL}/{INDEX_NAME}", headers=HEADERS)

response = requests.put(
    f"{BASE_URL}/{INDEX_NAME}", headers=HEADERS, data=json.dumps(mapping)
)
response.json()

{'acknowledged': True, 'shards_acknowledged': True, 'index': 'amazon-jp'}

In [8]:
response = requests.get(f"{BASE_URL}/{INDEX_NAME}/_mapping", headers=HEADERS)
response.json()

{'amazon-jp': {'mappings': {'properties': {'product_brand': {'type': 'text'},
    'product_bullet_point': {'type': 'text'},
    'product_color': {'type': 'text'},
    'product_description': {'type': 'text'},
    'product_id': {'type': 'keyword'},
    'product_locale': {'type': 'keyword'},
    'product_title': {'type': 'text'}}}}}

In [9]:
actions = df_products["product_id"].map(
    lambda x: json.dumps(
        {
            "index": {
                "_index": INDEX_NAME,
                "_id": x,
            }
        }
    )
)
# pandasの欠損値をJSONのnullへ変換する。NaNは正規のJSONではなく、
# OpenSearch Bulk APIでは該当itemだけが失敗する。
records = (
    df_products.astype(object)
    .where(pd.notna(df_products), None)
    .to_dict(orient="records")
)
data_jsons = [
    json.dumps(record, ensure_ascii=False, allow_nan=False) for record in records
]

jsonls = [
    action + "\n" + data_json + "\n" for action, data_json in zip(actions, data_jsons)
]
# 1度にすべて送信すると制限に引っかかるため10000件ずつ送信
chunk_size = 10000
for i in range(0, len(jsonls), chunk_size):
    jsonls_chunk = jsonls[i : i + chunk_size]
    print(f"Sending chunk {i} ~ {i + len(jsonls_chunk)}")

    res = requests.post(
        f"{BASE_URL}/{INDEX_NAME}/_bulk",
        headers={"Content-Type": "application/x-ndjson"},
        data="".join(jsonls_chunk),
    )
    res.raise_for_status()
    result = res.json()

    # Bulk APIは個別itemが失敗してもHTTP 200を返すため、
    # top-levelのerrorsと各itemのerrorを必ず確認する。
    if result.get("errors"):
        failed = [
            item["index"] for item in result["items"] if item["index"].get("error")
        ]
        first_error = failed[0]
        raise RuntimeError(
            f"Bulk indexing failed: {len(failed)} items in chunk starting at {i}. "
            f"First error: {json.dumps(first_error, ensure_ascii=False)}"
        )

print(f"Finished indexing {len(jsonls):,} products.")

Sending chunk 0 ~ 10000
Sending chunk 10000 ~ 20000
Sending chunk 20000 ~ 30000
Sending chunk 30000 ~ 40000
Sending chunk 40000 ~ 50000
Sending chunk 50000 ~ 60000
Sending chunk 60000 ~ 70000
Sending chunk 70000 ~ 80000
Sending chunk 80000 ~ 90000
Sending chunk 90000 ~ 100000
Sending chunk 100000 ~ 110000
Sending chunk 110000 ~ 120000
Sending chunk 120000 ~ 130000
Sending chunk 130000 ~ 140000
Sending chunk 140000 ~ 150000
Sending chunk 150000 ~ 160000
Sending chunk 160000 ~ 170000
Sending chunk 170000 ~ 180000
Sending chunk 180000 ~ 190000
Sending chunk 190000 ~ 200000
Sending chunk 200000 ~ 210000
Sending chunk 210000 ~ 220000
Sending chunk 220000 ~ 230000
Sending chunk 230000 ~ 240000
Sending chunk 240000 ~ 250000
Sending chunk 250000 ~ 260000
Sending chunk 260000 ~ 270000
Sending chunk 270000 ~ 280000
Sending chunk 280000 ~ 290000
Sending chunk 290000 ~ 300000
Sending chunk 300000 ~ 310000
Sending chunk 310000 ~ 320000
Sending chunk 320000 ~ 330000
Sending chunk 330000 ~ 339059
Fin

In [10]:
response = requests.get(
    f"{BASE_URL}/{INDEX_NAME}/_count",
    headers=HEADERS,
)
response.json()

{'count': 339059,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}}

## 検索してみる

In [ ]:
response = requests.get(
    f"{BASE_URL}/{INDEX_NAME}/_search",
    headers=HEADERS,
    params={"q": "product_title:iphone"},
)
response.json()

{'took': 6,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 10000, 'relation': 'gte'},
  'max_score': 1.0,
  'hits': [{'_index': 'amazon-jp',
    '_id': 'B06WVHDWVR',
    '_score': 1.0,
    '_source': {'product_id': 'B06WVHDWVR',
     'product_title': '(エイトトウキョウ)eight tokyo サングラス スタイリッシュ スクエア メンズ レディース 運転用 スポーツ ブルーライトカット メガネ 伊達メガネ [ 鯖江メーカー企画 ] ブラック/ライトブルー 2061-3',
     'product_description': '【おすすめＰＯＩＮＴ】<br> ■紫外線カット率99パーセント以上。あなたの目と肌をお守りします<br> ■本場、鯖江メーカーが企画しています<br> ■キズの付きにくいアクリルレンズ使用<br> ■フレームには軽くて丈夫なポリカーボネートとニッケル素材を使用。<br> ■男女兼用のおしゃれなフレームです<br> ■日本人の骨格に合わせて設計したフレームの為、フィット感抜群！<br>',
     'product_bullet_point': 'eight tokyoは株式会社エイトの商標登録アイウェアブランドです。紫外線（ＵＶ）カット率99パーセント以上。あなたの目と肌をお守りします。\nメガネの本場、鯖江メーカーと共に企画しています。キズの付きにくいアクリルレンズ使用。■付属品【TEIJIN社製の最先端繊維 オリジナル眼鏡拭き】【国産メルトン素材 ソフトケース】\n男女兼用のおしゃれなフレームです。日本人の骨格に合わせて設計したフレームの為、フィット感抜群！\nサイズ\u3000幅14cm高さ3.6cmテンプル長さ14.5cm\n【eight tokyo】流行の発信源、東京で誕生したeye wear ブランド\u3000サングラスを中心に数多

In [12]:
query_dsl = {
    "query": {
        # "match": {
        #   "product_title": "iphone"
        # }
        "match_all": {}
    }
}

response = requests.post(
    f"{BASE_URL}/{INDEX_NAME}/_search", headers=HEADERS, data=json.dumps(query_dsl)
)
response.json()

{'took': 16,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 10000, 'relation': 'gte'},
  'max_score': 1.0,
  'hits': [{'_index': 'amazon-jp',
    '_id': 'B06WVHDWVR',
    '_score': 1.0,
    '_source': {'product_id': 'B06WVHDWVR',
     'product_title': '(エイトトウキョウ)eight tokyo サングラス スタイリッシュ スクエア メンズ レディース 運転用 スポーツ ブルーライトカット メガネ 伊達メガネ [ 鯖江メーカー企画 ] ブラック/ライトブルー 2061-3',
     'product_description': '【おすすめＰＯＩＮＴ】<br> ■紫外線カット率99パーセント以上。あなたの目と肌をお守りします<br> ■本場、鯖江メーカーが企画しています<br> ■キズの付きにくいアクリルレンズ使用<br> ■フレームには軽くて丈夫なポリカーボネートとニッケル素材を使用。<br> ■男女兼用のおしゃれなフレームです<br> ■日本人の骨格に合わせて設計したフレームの為、フィット感抜群！<br>',
     'product_bullet_point': 'eight tokyoは株式会社エイトの商標登録アイウェアブランドです。紫外線（ＵＶ）カット率99パーセント以上。あなたの目と肌をお守りします。\nメガネの本場、鯖江メーカーと共に企画しています。キズの付きにくいアクリルレンズ使用。■付属品【TEIJIN社製の最先端繊維 オリジナル眼鏡拭き】【国産メルトン素材 ソフトケース】\n男女兼用のおしゃれなフレームです。日本人の骨格に合わせて設計したフレームの為、フィット感抜群！\nサイズ\u3000幅14cm高さ3.6cmテンプル長さ14.5cm\n【eight tokyo】流行の発信源、東京で誕生したeye wear ブランド\u3000サングラスを中心に数

In [14]:
query_dsl = {
    "query": {
        "bool": {
            "must": [
                {"match": {"product_title": "充電器"}},
                {"match": {"product_brand": "エレコム"}},
            ]
        }
    }
}